# Feature Exploration — All Emotions

Look at each Librosa feature **across every emotion at once** so we can eyeball which ones actually separate them. Picks up where `Librosa.ipynb` left off (one feature, one Angry clip) and adds the per-emotion comparison view.

Builds the same emotion + source dataframe used in `NewSearch.ipynb`, then walks through waveform, mel-spec, MFCC, RMS energy, pitch, spectral centroid, ZCR, and chroma — one section per feature.

In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, display

EMO_DIR = Path("../Emotions")

## 1. Build the file index

Same `source_of` rules as `NewSearch.ipynb`. We tag each clip with its source so we can skip SAVEE later — its files are mislabeled by folder (the speakers got dumped into the wrong emotion folders), so we don't want to pick one as the "representative" clip for its folder name.

In [2]:
def source_of(name):
    #Same patterns as NewSearch.ipynb
    if re.match(r"^\d+-\d+-", name):             return "RAVDESS"
    if re.match(r"^\d{4}_", name):                return "CREMA-D"
    if re.match(r"^(OAF|YAF|OA)_", name):          return "TESS"
    if re.match(r"^[a-z]{1,2}\d+\.wav$", name):  return "SAVEE"
    return "other"

rows = []
for emo_path in sorted(p for p in EMO_DIR.iterdir() if p.is_dir()):
    for wav in sorted(emo_path.glob("*.wav")):
        rows.append({"filename": wav.name, "emotion": emo_path.name,
                     "source": source_of(wav.name), "path": str(wav)})
df = pd.DataFrame(rows)
print(f"{len(df)} clips, {df['emotion'].nunique()} emotions")
df["emotion"].value_counts()

FileNotFoundError: [Errno 2] No such file or directory: '../Emotions'

## 2. Pick one representative clip per emotion

One non-SAVEE clip per emotion folder. We load each at the librosa default (22,050 Hz, mono) and stash them in a dict so every feature cell below can reuse them.

In [ ]:
# Take the first non-SAVEE clip from each emotion folder
picks = (df[df["source"] != "SAVEE"]
         .groupby("emotion", group_keys=False)
         .first()
         .reset_index())

clips = {}
for _, row in picks.iterrows():
    y, sr = librosa.load(row["path"])     # default sr=22050, mono
    clips[row["emotion"]] = (y, sr)
    print(f'{row["emotion"]:<10} {row["source"]:<8} {row["filename"]}  ({librosa.get_duration(y=y, sr=sr):.2f}s)')

EMO_ORDER = list(clips.keys())            # used to order subplots consistently below

Quick listen — play the chosen clip for each emotion to confirm the picks are reasonable before we plot.

In [ ]:
for emo, (y, sr) in clips.items():
    print(emo)
    display(Audio(data=y, rate=sr))

## 3. Waveform per emotion

Raw amplitude over time. Angry/Surprised tend to be loud with sharp bursts, Sad/Neutral flatter.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 11))
for ax, emo in zip(axes.ravel(), EMO_ORDER):
    y, sr = clips[emo]
    librosa.display.waveshow(y, sr=sr, ax=ax)
    ax.set_title(emo); ax.set_ylabel("amp")
axes.ravel()[-1].axis("off")              # 7 emotions, 8 slots — hide the empty one
fig.suptitle("Waveform per emotion"); plt.tight_layout(); plt.show()

## 4. Mel-spectrogram per emotion

Where the energy sits in frequency over time. This is the surface MFCCs are computed from — useful to see before looking at the MFCCs themselves.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 11))
for ax, emo in zip(axes.ravel(), EMO_ORDER):
    y, sr = clips[emo]
    S_db = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr), ref=np.max)
    librosa.display.specshow(S_db, sr=sr, x_axis="time", y_axis="mel", ax=ax)
    ax.set_title(emo)
axes.ravel()[-1].axis("off")
fig.suptitle("Mel-spectrogram per emotion (dB)"); plt.tight_layout(); plt.show()

## 5. MFCCs per emotion

The timbre fingerprint. The pattern across coefficients changes from emotion to emotion — that joint pattern is what the CNN will key off in `MyPick.ipynb`.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 11))
for ax, emo in zip(axes.ravel(), EMO_ORDER):
    y, sr = clips[emo]
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)    # 13 coeffs is plenty to eyeball
    librosa.display.specshow(mfccs, sr=sr, x_axis="time", ax=ax)
    ax.set_title(emo); ax.set_ylabel("MFCC")
axes.ravel()[-1].axis("off")
fig.suptitle("MFCCs per emotion"); plt.tight_layout(); plt.show()

## 6. RMS energy — all emotions overlaid

Loudness over time, one line per emotion. The cheapest, clearest arousal cue — high-arousal emotions ride higher.

In [ ]:
plt.figure(figsize=(11, 5))
for emo in EMO_ORDER:
    y, sr = clips[emo]
    rms = librosa.feature.rms(y=y)[0]                      # energy per frame
    t = librosa.frames_to_time(range(len(rms)), sr=sr)
    plt.plot(t, rms, label=emo, lw=2)
plt.xlabel("time (s)"); plt.ylabel("RMS energy")
plt.title("RMS energy per emotion"); plt.legend(); plt.tight_layout(); plt.show()

## 7. Pitch (F0) — all emotions overlaid

Voiced pitch over time (pyin). Line breaks are unvoiced frames. Tracks arousal too — angry/surprised/fearful sit higher than sad/neutral.

In [ ]:
plt.figure(figsize=(11, 5))
for emo in EMO_ORDER:
    y, sr = clips[emo]
    f0, voiced_flag, _ = librosa.pyin(y, fmin=65, fmax=400, sr=sr, hop_length=1024)
    t = librosa.frames_to_time(range(len(f0)), sr=sr, hop_length=1024)
    plt.plot(t, f0, label=emo, lw=2)                       # NaNs (unvoiced) show as gaps
plt.xlabel("time (s)"); plt.ylabel("F0 (Hz)")
plt.title("Pitch contour per emotion"); plt.legend(); plt.tight_layout(); plt.show()

## 8. Spectral centroid — all emotions overlaid

Brightness over time. Tense/bright voices push it up; duller ones sit lower.

In [ ]:
plt.figure(figsize=(11, 5))
for emo in EMO_ORDER:
    y, sr = clips[emo]
    cen = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    t = librosa.frames_to_time(range(len(cen)), sr=sr)
    plt.plot(t, cen, label=emo, lw=2)
plt.xlabel("time (s)"); plt.ylabel("centroid (Hz)")
plt.title("Spectral centroid per emotion"); plt.legend(); plt.tight_layout(); plt.show()

## 9. Zero-crossing rate — all emotions overlaid

Rough noisiness. Weaker separator on its own — included so we can see that for ourselves.

In [ ]:
plt.figure(figsize=(11, 5))
for emo in EMO_ORDER:
    y, sr = clips[emo]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    t = librosa.frames_to_time(range(len(zcr)), sr=sr)
    plt.plot(t, zcr, label=emo, lw=2)
plt.xlabel("time (s)"); plt.ylabel("ZCR")
plt.title("Zero-crossing rate per emotion"); plt.legend(); plt.tight_layout(); plt.show()

## 10. Chroma per emotion

12 pitch classes. A music feature — watch how little the pattern changes between emotions, evidence we can probably drop it from the feature stack.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 11))
for ax, emo in zip(axes.ravel(), EMO_ORDER):
    y, sr = clips[emo]
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    librosa.display.specshow(chroma, sr=sr, x_axis="time", y_axis="chroma", cmap="coolwarm", ax=ax)
    ax.set_title(emo)
axes.ravel()[-1].axis("off")
fig.suptitle("Chromagram per emotion"); plt.tight_layout(); plt.show()